In [1]:
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

In [2]:
X,y = make_classification(n_samples=10000, n_features=10,n_informative=3)

In [3]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [4]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train,y_train)
y_pred = dt.predict(X_test)

print("Decision Tree accuracy",accuracy_score(y_test,y_pred))

Decision Tree accuracy 0.925


In [6]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.5,
    bootstrap=True,
    random_state=42
)

In [8]:
bag.fit(X_train, y_train)

y_pred_bag = bag.predict(X_test)
print("Bagging Classifier Accuracy:", accuracy_score(y_test, y_pred_bag))

Bagging Classifier Accuracy: 0.9545


In [9]:
print("Rows given to Tree 1:", bag.estimators_samples_[0].shape)
print("Features (columns) given to Tree 1:", bag.estimators_features_[0].shape)

Rows given to Tree 1: (4000,)
Features (columns) given to Tree 1: (10,)


In [10]:
from sklearn.svm import SVC


bag_svm = BaggingClassifier(
    estimator=SVC(),
    n_estimators=500,
    max_samples=0.25,  # Each SVM gets only 25% of training data (keeps it fast)
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

#  Train
bag_svm.fit(X_train, y_train)

# Evaluate
y_pred_svm = bag_svm.predict(X_test)
print("Bagging (SVM) Accuracy:", accuracy_score(y_test, y_pred_svm))

Bagging (SVM) Accuracy: 0.9565


In [11]:
# Pasting: Exactly same as Bagging but bootstrap=False (no duplicate rows)
bag_pasting = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=False,   # <-- The ONLY difference from Bagging!
    random_state=42,
    n_jobs=-1
)

# Train and evaluate
bag_pasting.fit(X_train, y_train)
y_pred_pasting = bag_pasting.predict(X_test)
print("Pasting Classifier Accuracy:", accuracy_score(y_test, y_pred_pasting))

Pasting Classifier Accuracy: 0.956


In [12]:
# Random Subspaces: Sample COLUMNS instead of rows
bag_subspace = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=500,
    max_samples=1.0,           # All rows (no row sampling)
    bootstrap=False,           # No duplicate rows
    max_features=0.5,          # Only 50% of columns given to each tree
    bootstrap_features=True,   # Columns sampled with replacement
    random_state=42,
    n_jobs=-1
)

# Train and evaluate
bag_subspace.fit(X_train, y_train)
y_pred_subspace = bag_subspace.predict(X_test)
print("Random Subspaces Accuracy:", accuracy_score(y_test, y_pred_subspace))

# Peek inside: How many features did Tree 1 actually get?
print("Total features in dataset:", X_train.shape[1])
print("Features given to Tree 1:", bag_subspace.estimators_features_[0])

Random Subspaces Accuracy: 0.9405
Total features in dataset: 10
Features given to Tree 1: [9 2 9 7 7]


In [ ]:
# Random Patches: Row sampling AND Column sampling simultaneously
bag_patches = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=500,
    max_samples=0.25,          # Only 25% of rows
    bootstrap=True,            # Rows with replacement (duplicates allowed)
    max_features=0.5,          # Only 50% of columns
    bootstrap_features=True,   # Columns with replacement
    random_state=42,
    n_jobs=-1
)

# Train and evaluate
bag_patches.fit(X_train, y_train)
y_pred_patches = bag_patches.predict(X_test)
print("Random Patches Accuracy:", accuracy_score(y_test, y_pred_patches))

# Summary comparison
print("\n--- So Far accuracies ---")
print("Single Decision Tree  :", accuracy_score(y_test, dt.predict(X_test)))
print("Bagging (DT)          :", accuracy_score(y_test, y_pred_bag))
print("Pasting (DT)          :", accuracy_score(y_test, y_pred_pasting))
print("Random Subspaces (DT) :", accuracy_score(y_test, y_pred_subspace))
print("Random Patches (DT)   :", accuracy_score(y_test, y_pred_patches))

Random Patches Accuracy: 0.9465

--- Summary So Far ---
Single Decision Tree  : 0.925
Bagging (DT)          : 0.9545
Pasting (DT)          : 0.956
Random Subspaces (DT) : 0.9405
Random Patches (DT)   : 0.9465


In [14]:
# OOB Score: Uses the "leftover" rows (not in any bag) as automatic validation set
bag_oob = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    oob_score=True,    # <-- Enable OOB scoring
    random_state=42,
    n_jobs=-1
)

# Train
bag_oob.fit(X_train, y_train)

# OOB Score (calculated on leftover rows during training - NO test set needed!)
print("OOB Score (Auto-Validation):", bag_oob.oob_score_)

# Compare with actual test set accuracy
y_pred_oob = bag_oob.predict(X_test)
print("Actual Test Set Accuracy:   ", accuracy_score(y_test, y_pred_oob))

OOB Score (Auto-Validation): 0.953625
Actual Test Set Accuracy:    0.953


In [15]:
from sklearn.model_selection import GridSearchCV

# Define all parameter combinations to test
parameters = {
    'n_estimators'  : [50, 100, 500],
    'max_samples'   : [0.1, 0.4, 0.7, 1.0],
    'bootstrap'     : [True, False],
    'max_features'  : [0.1, 0.4, 0.7, 1.0]
}

# GridSearchCV will automatically test all combinations with 5-fold CV
search = GridSearchCV(
    BaggingClassifier(random_state=42, n_jobs=-1),
    parameters,
    cv=5,
    n_jobs=-1
)

# Train
search.fit(X_train, y_train)

# Best results
print("Best Parameters:", search.best_params_)
print("Best CV Score:  ", search.best_score_)

KeyboardInterrupt: 